In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 1993
month = 2


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1993-02-28


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1993-02-01 12:00:00
end_date 1993-02-02 12:00:00
start_date 1993-02-03 12:00:00
end_date 1993-02-04 12:00:00
start_date 1993-02-05 12:00:00
end_date 1993-02-06 12:00:00
start_date 1993-02-07 12:00:00
end_date 1993-02-08 12:00:00
start_date 1993-02-09 12:00:00
end_date 1993-02-10 12:00:00
start_date 1993-02-11 12:00:00
end_date 1993-02-12 12:00:00
start_date 1993-02-13 12:00:00
end_date 1993-02-14 12:00:00
start_date 1993-02-15 12:00:00
end_date 1993-02-16 12:00:00
start_date 1993-02-17 12:00:00
end_date 1993-02-18 12:00:00
start_date 1993-02-19 12:00:00
end_date 1993-02-20 12:00:00
start_date 1993-02-21 12:00:00
end_date 1993-02-22 12:00:00
start_date 1993-02-23 12:00:00
end_date 1993-02-24 12:00:00
start_date 1993-02-25 12:00:00
end_date 1993-02-26 12:00:00
start_date 1993-02-27 12:00:00
end_date 1993-02-28 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                                                                                           | 0/14 [00:00<?, ?it/s]

  7%|████████▏                                                                                                         | 1/14 [03:21<43:38, 201.45s/it]

 14%|████████████████▍                                                                                                  | 2/14 [03:41<18:54, 94.56s/it]

 21%|████████████████████████▋                                                                                          | 3/14 [04:05<11:28, 62.62s/it]

 29%|████████████████████████████████▊                                                                                  | 4/14 [04:37<08:25, 50.55s/it]

...

 86%|█████████████████████████████████████████████████████████████████████████████████████████████████▋                | 12/14 [07:42<00:45, 22.72s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 13/14 [08:03<00:22, 22.04s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [08:24<00:00, 21.70s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [08:24<00:00, 36.03s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesU_1993-02.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                                                                                           | 0/14 [00:00<?, ?it/s]

  7%|████████▏                                                                                                         | 1/14 [02:22<30:46, 142.01s/it]

 14%|████████████████▍                                                                                                  | 2/14 [03:13<17:42, 88.58s/it]

 21%|████████████████████████▋                                                                                          | 3/14 [03:39<11:04, 60.37s/it]

 29%|████████████████████████████████▊                                                                                  | 4/14 [04:39<10:01, 60.18s/it]

...

 86%|█████████████████████████████████████████████████████████████████████████████████████████████████▋                | 12/14 [10:39<01:09, 34.70s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 13/14 [11:00<00:30, 30.45s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [11:26<00:00, 29.33s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [11:26<00:00, 49.07s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesV_1993-02.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                                                                                           | 0/14 [00:00<?, ?it/s]

  7%|████████▏                                                                                                         | 1/14 [02:07<27:39, 127.62s/it]

 14%|████████████████▍                                                                                                  | 2/14 [02:27<12:47, 63.97s/it]

 21%|████████████████████████▋                                                                                          | 3/14 [02:46<08:00, 43.66s/it]

 29%|████████████████████████████████▊                                                                                  | 4/14 [03:08<05:49, 34.96s/it]

...

 86%|█████████████████████████████████████████████████████████████████████████████████████████████████▋                | 12/14 [08:30<00:58, 29.35s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 13/14 [08:51<00:26, 26.80s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [09:12<00:00, 25.01s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [09:12<00:00, 39.44s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesW_1993-02.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                                                                                           | 0/14 [00:00<?, ?it/s]

  7%|████████▏                                                                                                          | 1/14 [01:07<14:32, 67.10s/it]

 14%|████████████████▍                                                                                                  | 2/14 [01:25<07:38, 38.25s/it]

 21%|████████████████████████▋                                                                                          | 3/14 [01:50<05:54, 32.26s/it]

 29%|████████████████████████████████▊                                                                                  | 4/14 [02:08<04:28, 26.88s/it]

...

 86%|█████████████████████████████████████████████████████████████████████████████████████████████████▋                | 12/14 [05:15<00:45, 22.57s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 13/14 [05:39<00:22, 22.80s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [05:57<00:00, 21.36s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [05:57<00:00, 25.52s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesT_1993-02.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                                                                                           | 0/14 [00:00<?, ?it/s]

  7%|████████▏                                                                                                         | 1/14 [01:46<23:09, 106.89s/it]

 14%|████████████████▍                                                                                                  | 2/14 [02:03<10:46, 53.83s/it]

 21%|████████████████████████▋                                                                                          | 3/14 [03:32<12:47, 69.74s/it]

 29%|████████████████████████████████▊                                                                                  | 4/14 [03:58<08:44, 52.46s/it]

...

 86%|█████████████████████████████████████████████████████████████████████████████████████████████████▋                | 12/14 [07:55<00:52, 26.29s/it]

 93%|█████████████████████████████████████████████████████████████████████████████████████████████████████████▊        | 13/14 [08:13<00:23, 23.64s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [08:32<00:00, 22.37s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [08:32<00:00, 36.62s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variablesS_1993-02.nc
